# ICU Handoff Summary Evaluation

Generated summaries (`output/results/P{N} summary.txt`) vs Gold Standard (`3단계_ Gold Standard Summary.xlsx`)

**Metrics**: BERTScore (Precision, Recall, F1), SBERT Cosine Similarity

In [ ]:
# === Clone repo (Colab only) ===
import os
if not os.path.exists("emr_icu_handover"):
    !git clone https://github.com/taejun-song/icu_nurse_handoff.git emr_icu_handover
os.chdir("emr_icu_handover")

In [ ]:
!pip install -q bert-score sentence-transformers

In [ ]:
# === Configuration ===
GOLD_XLSX_PATH = "3단계_ Gold Standard Summary.xlsx"  # relative to repo root
PRED_DIR = "output/results"                           # directory with P{N} summary.txt
OUTPUT_DIR = "output/eval_results"                    # where to save evaluation outputs

USE_SBERT = True  # set False to skip SBERT (saves time/memory)

BERTSCORE_MODEL = "xlm-roberta-large"
SBERT_MODEL = "jhgan/ko-sroberta-multitask"

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "src").exists(), f"src/ not found in {REPO_ROOT}. Run the clone cell first."

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import display, Markdown
from src.alignment import load_gold_standard, load_predictions, parse_txt_sections, build_pairs
from src.eval import (
    compute_bertscore, compute_sbert_cosine,
    aggregate_by_sheet, aggregate_by_framework, aggregate_by_level2,
    aggregate_overall, save_results,
)

GOLD_XLSX = REPO_ROOT / GOLD_XLSX_PATH
PRED = REPO_ROOT / PRED_DIR
OUT = REPO_ROOT / OUTPUT_DIR
print(f"Repo root: {REPO_ROOT}")
print(f"Gold XLSX: {GOLD_XLSX} (exists={GOLD_XLSX.exists()})")
print(f"Pred dir:  {PRED} (exists={PRED.exists()})")

In [ ]:
# === Inspect gold standard ===
gold_sheets = load_gold_standard(GOLD_XLSX)
print(f"Patient sheets: {list(gold_sheets.keys())} ({len(gold_sheets)} total)\n")
for name, df in gold_sheets.items():
    print(f"  {name}: {len(df)} rows, columns={list(df.columns[:3])}")

In [ ]:
# === Inspect predictions ===
predictions = load_predictions(PRED)
print(f"Prediction files found: {len(predictions)}\n")
for pid, text in sorted(predictions.items(), key=lambda x: int(x[0][1:])):
    print(f"  {pid}: {len(text)} chars")

In [ ]:
# === Build alignment pairs ===
pairs_df, unmatched_gold, unmatched_pred = build_pairs(GOLD_XLSX, PRED)
print(f"Total aligned pairs: {len(pairs_df)}")
print(f"Unmatched gold sheets: {unmatched_gold}")
print(f"Unmatched pred files:  {unmatched_pred}")
print(f"\nAlignment methods: {pairs_df['alignment_method'].value_counts().to_dict()}")

In [ ]:
# === Preview aligned pairs ===
preview = pairs_df[["sheet_name", "level_2", "gold_text_norm", "pred_text_norm"]].copy()
preview["gold_text_norm"] = preview["gold_text_norm"].str[:200]
preview["pred_text_norm"] = preview["pred_text_norm"].str[:200]
with pd.option_context("display.max_colwidth", 200, "display.max_rows", 30):
    display(preview.head(30))

In [ ]:
# === Compute metrics ===
golds = pairs_df["gold_text_norm"].tolist()
preds = pairs_df["pred_text_norm"].tolist()

print("Computing BERTScore...")
bert_results = compute_bertscore(golds, preds, model_type=BERTSCORE_MODEL)
pairs_df["bertscore_precision"] = bert_results["precision"]
pairs_df["bertscore_recall"] = bert_results["recall"]
pairs_df["bertscore_f1"] = bert_results["f1"]
print(f"  done. P={pairs_df['bertscore_precision'].mean():.4f}  R={pairs_df['bertscore_recall'].mean():.4f}  F1={pairs_df['bertscore_f1'].mean():.4f}")

if USE_SBERT:
    print("Computing SBERT cosine similarity...")
    pairs_df["sbert_cosine"] = compute_sbert_cosine(golds, preds, model_name=SBERT_MODEL)
    print(f"  done. mean={pairs_df['sbert_cosine'].mean():.4f}")

print("\nAll metrics computed.")

In [ ]:
# === Per-sample results ===
metric_cols = [c for c in pairs_df.columns if c.startswith(("bertscore", "sbert"))]
display_cols = ["sheet_name", "level_1", "level_2"] + metric_cols
with pd.option_context("display.max_rows", 220, "display.float_format", "{:.4f}".format):
    display(pairs_df[display_cols])

In [ ]:
# === Per-sheet aggregation ===
sheet_agg = aggregate_by_sheet(pairs_df, metric_cols)
with pd.option_context("display.float_format", "{:.4f}".format):
    display(sheet_agg)

In [ ]:
# === Per-framework aggregation (from Google Sheets) ===
from io import StringIO
import requests
from src.alignment import LEVEL_2_TO_LEVEL_1, HEADER_ALIASES, normalize_text

EXTRACTED_SHEET_ID = "1UfCYsbWMITlHE_KkPeLlOwV8WUC7Y4Y_7kRgWelEggk"
GOLD_SHEET_ID = "1Sfm8CbjTXtLkb0NJwRA7mnRMIUmZOQFYdpT4Fg8KwW4"

def fetch_csv(sheet_id, sheet_name=None, gid=None):
    if sheet_name:
        url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
    elif gid is not None:
        url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    else:
        url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
    resp = requests.get(url)
    resp.raise_for_status()
    return pd.read_csv(StringIO(resp.text))

def normalize_df(df, has_no_col=False):
    if has_no_col:
        df.columns = ["No.", "Level 1", "Level 2", "Level 3"] + list(df.columns[4:])
        df["No."] = df["No."].ffill()
    else:
        df.columns = ["Level 1", "Level 2", "Level 3"] + list(df.columns[3:])
    df["Level 1"] = df["Level 1"].ffill()
    df["Level 2"] = df["Level 2"].ffill()
    df["Level 2"] = df["Level 2"].apply(
        lambda x: HEADER_ALIASES.get(str(x).strip(), str(x).strip()) if pd.notna(x) else x
    )
    return df

def build_sections(df):
    sections = {}
    for _, r in df.iterrows():
        l2 = str(r["Level 2"]).strip() if pd.notna(r["Level 2"]) else ""
        l3 = str(r["Level 3"]) if pd.notna(r["Level 3"]) else ""
        if l2 in sections:
            sections[l2] += "\n" + l3
        else:
            sections[l2] = l3
    return sections

extracted_df = fetch_csv(EXTRACTED_SHEET_ID)
extracted_df = normalize_df(extracted_df, has_no_col=True)
print(f"Extracted: {len(extracted_df)} rows, patients: {sorted(extracted_df['No.'].unique(), key=lambda x: int(x[1:]))}")

gold_dfs = {}
for i in range(1, 16):
    pid = f"P{i}"
    try:
        df = fetch_csv(GOLD_SHEET_ID, sheet_name=pid)
        gold_dfs[pid] = normalize_df(df, has_no_col=False)
    except Exception as e:
        print(f"  Skipped {pid}: {e}")
print(f"Gold standard: {len(gold_dfs)} patient sheets loaded")

rows = []
for pid, gold_df in sorted(gold_dfs.items(), key=lambda x: int(x[0][1:])):
    ext_patient = extracted_df[extracted_df["No."] == pid]
    if ext_patient.empty:
        print(f"  WARNING: No extracted data for {pid}")
        continue
    ext_sections = build_sections(ext_patient)
    gold_sections = build_sections(gold_df)
    for l2 in LEVEL_2_TO_LEVEL_1:
        gold_text = gold_sections.get(l2, "")
        pred_text = ext_sections.get(l2, "")
        rows.append({
            "sheet_name": pid,
            "level_1": LEVEL_2_TO_LEVEL_1[l2],
            "level_2": l2,
            "gold_text_norm": normalize_text(gold_text),
            "pred_text_norm": normalize_text(pred_text),
        })
fw_pairs_df = pd.DataFrame(rows)
print(f"\nTotal aligned pairs: {len(fw_pairs_df)}")
print(f"Pairs per framework:\n{fw_pairs_df.groupby('level_1').size().to_dict()}")

In [ ]:
# === Compute framework metrics & aggregate ===
fw_golds = fw_pairs_df["gold_text_norm"].tolist()
fw_preds = fw_pairs_df["pred_text_norm"].tolist()

print("Computing BERTScore...")
fw_bert = compute_bertscore(fw_golds, fw_preds, model_type=BERTSCORE_MODEL)
fw_pairs_df["bertscore_precision"] = fw_bert["precision"]
fw_pairs_df["bertscore_recall"] = fw_bert["recall"]
fw_pairs_df["bertscore_f1"] = fw_bert["f1"]
print(f"  done. P={fw_pairs_df['bertscore_precision'].mean():.4f}  R={fw_pairs_df['bertscore_recall'].mean():.4f}  F1={fw_pairs_df['bertscore_f1'].mean():.4f}")

if USE_SBERT:
    print("Computing SBERT cosine similarity...")
    fw_pairs_df["sbert_cosine"] = compute_sbert_cosine(fw_golds, fw_preds, model_name=SBERT_MODEL)
    print(f"  done. mean={fw_pairs_df['sbert_cosine'].mean():.4f}")

fw_metric_cols = [c for c in fw_pairs_df.columns if c.startswith(("bertscore", "sbert"))]
framework_agg = aggregate_by_framework(fw_pairs_df, fw_metric_cols)
with pd.option_context("display.float_format", "{:.4f}".format):
    display(framework_agg)
assert framework_agg["count"].sum() == len(fw_pairs_df), "Entry count mismatch"
print(f"\nValidation passed: {framework_agg['count'].sum()} entries across {len(framework_agg)} frameworks")

In [ ]:
# === Per-Level 2 aggregation ===
level2_agg = aggregate_by_level2(fw_pairs_df, fw_metric_cols)
with pd.option_context("display.float_format", "{:.4f}".format):
    display(level2_agg)
assert level2_agg["count"].sum() == len(fw_pairs_df), "Entry count mismatch"
print(f"\nValidation passed: {level2_agg['count'].sum()} entries across {len(level2_agg)} subcategories")

In [ ]:
# === Overall aggregation ===
overall = aggregate_overall(pairs_df, metric_cols)
for col in metric_cols:
    s = overall[col]
    print(f"{col}: mean={s['mean']:.4f}  median={s['median']:.4f}  std={s['std']:.4f}")
print(f"\nTotal pairs: {overall['n_pairs']}")

In [ ]:
# === Save outputs ===
config = {
    "gold_xlsx": str(GOLD_XLSX),
    "pred_dir": str(PRED),
    "bertscore_model": BERTSCORE_MODEL,
    "sbert_model": SBERT_MODEL if USE_SBERT else None,
    "use_sbert": USE_SBERT,
}
save_results(pairs_df, sheet_agg, overall, OUT, config=config, framework_agg=framework_agg, level2_agg=level2_agg)
print(f"Results saved to {OUT}/")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")